# Notebook 18 — bancada de triagem (screening rápido de tratamento de dados)

**Propósito:** testar ideias de tratamento de dados / feature engineering em minutos,
não horas. Uma só partição fixa, só XGB, um baseline e depois as variantes.
**Isto NÃO produz números de relatório** — só filtra o que merece o completo (5-fold).

**Protocolo fixo:**
- 1 holdout estratificado 80/20, seed 42, índices guardados em `data/nb18_split.npz`.
  Todas as variantes usam A MESMA partição — sem isto nada é comparável.
- Modelo único: XGB `n_estimators=500, max_depth=6, lr=0.05, subsample/colsample=0.8,
  reg_lambda=1.0` + `sample_weight='balanced'`. Sem tuning aqui.
- Métrica: Macro-F1 no holdout + tempo. Baseline corre primeiro; cada teste reporta
  o delta contra o baseline.

**Regras de promoção (contra o otimismo do holdout único, ±0.01–0.02 de ruído):**
- delta >= +0.010 no holdout → corre no completo (5-fold, NB08/NB12/NB17).
- Abaixo disso → arquivado com número. Sem exceções.
- Subsample (<100% linhas) SÓ para smoke de código, nunca para decisões: com 172 casos
  na classe 1, reduzir dados distorce as minoritárias e o ranking (lição do rapido 50%).


In [ ]:
import os, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
K, TARGET, RANDOM_STATE = 16, 'AGE_RANGE', 42
DATA_DIR = '../data' if os.path.isdir('../data') else 'data'
tr = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = tr[TARGET].to_numpy().astype(int)
Xv = tr.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
COLS = tr.drop(columns=['PID', TARGET]).columns.to_numpy()
sp = os.path.join(DATA_DIR, 'nb18_split.npz')
if os.path.exists(sp):
    z = np.load(sp); itr, ite = z['itr'], z['ite']
else:
    itr, ite = next(StratifiedShuffleSplit(1, test_size=0.2, random_state=RANDOM_STATE).split(Xv, y))
    np.savez(sp, itr=itr, ite=ite)
print('treino %d | holdout %d' % (len(itr), len(ite)))
def xgb_recipe(sd=42):
    return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', random_state=sd,
        n_jobs=-1, eval_metric='mlogloss')
def avaliar(nome, Xtr, Xte, nota='', ytr=None):
    t0 = time.time()
    yy = y[itr] if ytr is None else ytr
    m = xgb_recipe(); m.fit(Xtr, yy, sample_weight=compute_sample_weight('balanced', yy))
    p = m.predict(Xte)
    f = f1_score(y[ite], p, average='macro')
    _d = '' if 'BASE' not in globals() else ' | delta %+.4f' % (f - BASE)
    r = {'variante': nome, 'f1': round(float(f), 4), 'tempo_s': round(time.time() - t0, 1), 'nota': (nota + _d).strip()}
    RESULTADOS.append(r)
    print('%s: %.4f (%ds) %s' % (nome, f, r['tempo_s'], nota), flush=True)
    return f, p
RESULTADOS = []


## T0 — baseline (referência de todos os deltas)

Corre UMA vez. Se a base mudar (novas colunas no `train_nb02.csv`), re-corre e
atualiza o `BASE`.

In [ ]:
BASE, _ = avaliar('T0 baseline XGB', Xv[itr], Xv[ite])
pd.DataFrame(RESULTADOS).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('BASE =', round(float(BASE), 4))


## Modelo de célula de teste (copiar por ideia)

Cada ideia nova = uma cópia deste bloco com nome T<n>. Regras: usar `itr`/`ite`
(a partição fixa), o `xgb_recipe` sem alterações, e fechar com o `avaliar` +
re-gravar o CSV. Nada de tuning, nada de trocar a partição.


In [ ]:
# TPL — exemplo (não corre nada novo: repete a base para validar o harness)
_, _ = avaliar('TPL repeticao da base', Xv[itr], Xv[ite], nota='deve dar ~BASE')
pd.DataFrame(RESULTADOS).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
pd.DataFrame(RESULTADOS).sort_values('f1', ascending=False).to_string(index=False)


In [ ]:
# T1 — recusadores: N_SEM_RESPOSTA + flag>20 (revisita E11/§7.3 na base 311)
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
assert (pd.to_numeric(t01['PID']).to_numpy() == pd.to_numeric(tr['PID'].to_numpy())).all()
txt = t01.select_dtypes(include='str')
nref = ((txt == 'Sem resposta') | (txt == 'NR')).sum(axis=1).to_numpy().astype(float)
X1 = np.hstack([Xv, nref[:, None], (nref > 20).astype(float)[:, None]])
avaliar('T1 recusadores (N_SEM_RESPOSTA+flag)', X1[itr], X1[ite],
        nota='recusadores (N_SEM_RESPOSTA+flag)')

_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = pd.DataFrame([r for r in _ant.to_dict('records') if r['variante'] not in [x['variante'] for x in RESULTADOS[-1:]]])
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


In [ ]:
# T2 — correcao pos-previsao FAIXA_MINIMA (sem refit: segundos)
import time as _t
_t0 = _t.time()
_m = xgb_recipe(); _m.fit(Xv[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
_p = _m.predict(Xv[ite])
fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
_pc = np.maximum(_p, fmin[ite])
fcorr = f1_score(y[ite], _pc, average='macro')
RESULTADOS.append({'variante': 'T2 +correcao FAIXA_MINIMA', 'f1': round(float(fcorr), 4),
                   'tempo_s': round(_t.time() - _t0, 1),
                   'nota': 'delta %+.4f vs BASE; corrigidas %d' % (fcorr - BASE, int((_pc != _p).sum()))})
print('T2 +correcao FAIXA_MINIMA: %.4f' % fcorr, flush=True)

_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = pd.DataFrame([r for r in _ant.to_dict('records') if r['variante'] not in [x['variante'] for x in RESULTADOS[-1:]]])
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


In [ ]:
# T3 — ablacao: SEM IDADE_MINIMA/FAIXA_MINIMA como features (valor p/ relatorio)
keep = [c for c in COLS if c not in ('IDADE_MINIMA', 'FAIXA_MINIMA')]
X3 = tr[keep].to_numpy(dtype=np.float64)
avaliar('T3 sem IDADE_MINIMA/FAIXA_MINIMA', X3[itr], X3[ite],
        nota='sem IDADE_MINIMA/FAIXA_MINIMA')

_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = pd.DataFrame([r for r in _ant.to_dict('records') if r['variante'] not in [x['variante'] for x in RESULTADOS[-1:]]])
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


In [ ]:
# T4 — MI top-150 in-holdout (E21 renascida como triagem; paridade p/ velocidade)
from sklearn.feature_selection import mutual_info_classif
_mi = mutual_info_classif(Xv[itr], y[itr], random_state=RANDOM_STATE)
_top = np.argsort(_mi)[-150:]
avaliar('T4 MI top-150', Xv[itr][:, _top], Xv[ite][:, _top], nota='paridade p/ triagem')

_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = pd.DataFrame([r for r in _ant.to_dict('records') if r['variante'] not in [x['variante'] for x in RESULTADOS[-1:]]])
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


In [ ]:
# T5 — target-mean encoding (MARSTALEGAL/MAINSTAT/HATLEVEL) em vez de one-hots
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
assert (pd.to_numeric(t01['PID']).to_numpy() == pd.to_numeric(tr['PID'].to_numpy())).all()
_tm = np.zeros((len(tr), 3))
for j, c in enumerate(['MARSTALEGAL', 'MAINSTAT', 'HATLEVEL']):
    _map = pd.Series(y[itr]).groupby(t01[c].iloc[itr].values).mean()
    _tm[:, j] = t01[c].map(_map).fillna(_map.mean()).to_numpy()
_drop = [c for c in COLS if any(c.startswith(p + '_') for p in ['MARSTALEGAL', 'MAINSTAT'])]
_keep = [c for c in COLS if c not in _drop]
X5 = np.hstack([tr[_keep].to_numpy(dtype=np.float64), _tm])
avaliar('T5 target-mean MARSTALEGAL/MAINSTAT/HATLEVEL', X5[itr], X5[ite],
        nota='drop %d one-hots, +3 medias' % len(_drop))

_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = pd.DataFrame([r for r in _ant.to_dict('records') if r['variante'] not in [x['variante'] for x in RESULTADOS[-1:]]])
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


In [ ]:
# T6 — quarentena honesta drop-worst-10%: confianca inner-OOF sobre TODO o treino
# (4-fold interno; so depois drop 10% e refit). Tudo dentro do split de treino.
from sklearn.model_selection import StratifiedKFold as _SKF
_in = list(_SKF(4, shuffle=True, random_state=43).split(Xv[itr], y[itr]))
_conf = np.zeros(len(itr))
for _a, _b in _in:
    _mq = xgb_recipe()
    _mq.fit(Xv[itr][_a], y[itr][_a], sample_weight=compute_sample_weight('balanced', y[itr][_a]))
    _conf[_b] = _mq.predict_proba(Xv[itr][_b]).max(1)
_keep = np.argsort(_conf)[int(len(itr) * 0.10):]
avaliar('T6 quarentena drop-worst-10% (honesta)', Xv[itr][_keep], Xv[ite],
        nota='treino %d->%d' % (len(itr), len(_keep)), ytr=y[itr][_keep])
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T6 done', flush=True)


In [ ]:
# T7 — pairwise gated nos pares quentes, base 311 (E2 renascida no harness)
# Override so quando top-2 == par quente (hard gate). Precedente base antiga: +0.009.
from sklearn.utils import compute_sample_weight as _csw
QUENTES = [(3, 4), (4, 5), (5, 6), (6, 7), (12, 13), (13, 14)]
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_pb = _mb.predict_proba(Xv[ite]); _pa = _mb.predict(Xv[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_pred = _pa.copy(); _n_over = 0
for a_, b_ in QUENTES:
    mtr = np.isin(y[itr], [a_, b_])
    yb = (y[itr][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xv[itr][mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n_over += int(sel.sum())
_f7 = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'T7 pairwise gated (6 pares)', 'f1': round(float(_f7), 4),
                   'tempo_s': 0, 'nota': 'override %d linhas | delta %+.4f' % (_n_over, _f7 - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T7 pairwise gated: %.4f (override %d)' % (_f7, _n_over), flush=True)


In [ ]:
# T8 — drop bottom-30 por ganho do XGB (ranking gratis; o teste honesto e o refit)
_m8 = xgb_recipe(); _m8.fit(Xv[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
_keep8 = np.argsort(-_m8.feature_importances_)[:-30]
_m8b = xgb_recipe(); _m8b.fit(Xv[itr][:, _keep8], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
_f8 = f1_score(y[ite], _m8b.predict(Xv[ite][:, _keep8]), average='macro')
RESULTADOS.append({'variante': 'T8 drop bottom-30 ganho', 'f1': round(float(_f8), 4),
                   'tempo_s': 0, 'nota': 'delta %+.4f' % (_f8 - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T8 drop bottom-30: %.4f' % _f8, flush=True)


In [ ]:
# C1 — combo mecanistico: SEM floor-features (T3) + correcao pos-previsao (T2)
# Pergunta: a floor e redundante como feature se existir como correcao? (ortogonais)
keep = [c for c in COLS if c not in ('IDADE_MINIMA', 'FAIXA_MINIMA')]
Xc = tr[keep].to_numpy(dtype=np.float64)
_mc = xgb_recipe(); _mc.fit(Xc[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
_pc = _mc.predict(Xc[ite])
fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
_pcc = np.maximum(_pc, fmin[ite])
_fc = f1_score(y[ite], _pcc, average='macro')
RESULTADOS.append({'variante': 'C1 sem-floor + correcao', 'f1': round(float(_fc), 4),
                   'tempo_s': 0, 'nota': 'delta %+.4f | corrigidas %d' % (_fc - BASE, int((_pcc != _pc).sum()))})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('C1 sem-floor + correcao: %.4f' % _fc, flush=True)


In [ ]:
# C2 — combo mecanistico: target-mean (T5) + pairwise gated (T7)
# Representacao x decisao: mecanismos diferentes, a unica sinergia defensavel.
# SO corre se T7 mostrar sinal (gate: T7 >= +0.008); senao e fishing.
import pandas as _pd
_h = _pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_t7 = _h.loc[_h['variante'].str.startswith('T7 '), 'f1']
_gate = (float(_t7.max()) - BASE) if len(_t7) else -9.0
print('T7 delta: %+.4f (gate +0.008)' % _gate, flush=True)
if _gate >= 0.008:
    t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
    _tm = np.zeros((len(tr), 3))
    for j_, c_ in enumerate(['MARSTALEGAL', 'MAINSTAT', 'HATLEVEL']):
        _map = pd.Series(y[itr]).groupby(t01[c_].iloc[itr].values).mean()
        _tm[:, j_] = t01[c_].map(_map).fillna(_map.mean()).to_numpy()
    _drop = [c for c in COLS if any(c.startswith(p + '_') for p in ['MARSTALEGAL', 'MAINSTAT'])]
    _keep = [c for c in COLS if c not in _drop]
    X5c = np.hstack([tr[_keep].to_numpy(dtype=np.float64), _tm])
    _mb = xgb_recipe(); _mb.fit(X5c[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
    _pb = _mb.predict_proba(X5c[ite]); _pa = _mb.predict(X5c[ite])
    _top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
    _pred = _pa.copy(); _n = 0
    for a_, b_ in [(3, 4), (4, 5), (5, 6), (6, 7), (12, 13), (13, 14)]:
        mtr = np.isin(y[itr], [a_, b_]); yb = (y[itr][mtr] == b_).astype(int)
        m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, min_child_weight=3,
                          subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
        m.fit(X5c[itr][mtr], yb, sample_weight=compute_sample_weight('balanced', yb))
        sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
        if sel.sum():
            _pred[sel] = np.where(m.predict_proba(X5c[ite][sel])[:, 1] >= 0.5, b_, a_)
            _n += int(sel.sum())
    _f2 = f1_score(y[ite], _pred, average='macro')
    RESULTADOS.append({'variante': 'C2 target-mean + pairwise', 'f1': round(float(_f2), 4),
                       'tempo_s': 0, 'nota': 'delta %+.4f | override %d' % (_f2 - BASE, _n)})
    _ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
    _ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
    pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
    print('C2 target-mean + pairwise: %.4f' % _f2, flush=True)
else:
    print('C2 saltado: T7 sem sinal, combo seria fishing', flush=True)


In [ ]:
# T9 — target-mean SUAVIZADO (m=100): refino da T5 (+0.0050 sem smoothing)
# T5 usou medias cruas -> categorias raras sobreajustam. m-estimate puxa-as ao global.
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
assert (pd.to_numeric(t01['PID']).to_numpy() == pd.to_numeric(tr['PID'].to_numpy())).all()
_glob = y[itr].mean(); _m = 100.0
_tm = np.zeros((len(tr), 3))
for j_, c_ in enumerate(['MARSTALEGAL', 'MAINSTAT', 'HATLEVEL']):
    _g = pd.Series(y[itr]).groupby(t01[c_].iloc[itr].values)
    _sm = ((_g.sum() + _m * _glob) / (_g.size() + _m))
    _tm[:, j_] = t01[c_].map(_sm).fillna(_glob).to_numpy()
_drop = [c for c in COLS if any(c.startswith(p + '_') for p in ['MARSTALEGAL', 'MAINSTAT'])]
_keep = [c for c in COLS if c not in _drop]
X9 = np.hstack([tr[_keep].to_numpy(dtype=np.float64), _tm])
avaliar('T9 target-mean suavizado m=100', X9[itr], X9[ite])
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T9 done', flush=True)


In [ ]:
# T10 — pairwise gated nos 15 pares (T7 usou so os 6 quentes)
from sklearn.utils import compute_sample_weight as _csw
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_pb = _mb.predict_proba(Xv[ite]); _pa = _mb.predict(Xv[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(k, k + 1) for k in range(15)]:
    mtr = np.isin(y[itr], [a_, b_]); yb = (y[itr][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xv[itr][mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_f10 = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'T10 pairwise gated (15 pares)', 'f1': round(float(_f10), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f' % (_n, _f10 - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T10 pairwise 15 pares: %.4f (override %d)' % (_f10, _n), flush=True)


In [ ]:
# T11 — teto FAIXA_MAXIMA (simetrico da correcao de piso; regra de rejeicao tipo IN25)
# So Aluno tem teto limpo: verifica violacoes no treino, aborta se >1% (como IN25).
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
_al = (t01['MAINSTAT'] == 'Aluno, estudante, ou em estágio curricular').to_numpy()
_viol = ((y[itr] > 5) & _al[itr]).sum() / max(_al[itr].sum(), 1)
print('alunos no treino: %d | violacoes teto 5: %.3f' % (int(_al[itr].sum()), _viol), flush=True)
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
_p11 = _mb.predict(Xv[ite])
if _viol <= 0.01:
    _al_te = (t01['MAINSTAT'].iloc[ite].values == 'Aluno, estudante, ou em estágio curricular')
    _p11c = np.minimum(_p11, np.where(_al_te, 5, 15))
    _f11 = f1_score(y[ite], _p11c, average='macro')
    _nota = 'teto Aluno=5 | corrigidas %d | delta %+.4f' % (int((_p11c != _p11).sum()), _f11 - BASE)
else:
    _f11 = f1_score(y[ite], _p11, average='macro')
    _nota = 'REJEITADA: violacoes %.3f > 1%% (criterio IN25)' % _viol
RESULTADOS.append({'variante': 'T11 teto FAIXA_MAXIMA', 'f1': round(float(_f11), 4), 'tempo_s': 0, 'nota': _nota})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T11: %.4f' % _f11, flush=True)


In [ ]:
# T12 — restricoes monotonicas (conhecimento de dominio como prior, nao tuning):
# doenca/dificuldades/pisos sobem com a idade; escolaridade desce (efeito coorte).
_mc = {c: 0 for c in COLS}
for c in ['N_DOENCAS', 'N_DIFICULDADES', 'IDADE_MINIMA', 'FAIXA_MINIMA']:
    if c in _mc: _mc[c] = 1
if 'HATLEVEL' in _mc: _mc['HATLEVEL'] = -1
_mono = tuple(_mc[c] for c in COLS)
_m12 = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                     monotone_constraints=_mono, random_state=RANDOM_STATE,
                     n_jobs=-1, eval_metric='mlogloss')
_m12.fit(Xv[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
_f12 = f1_score(y[ite], _m12.predict(Xv[ite]), average='macro')
RESULTADOS.append({'variante': 'T12 restricoes monotonicas', 'f1': round(float(_f12), 4),
                   'tempo_s': 0, 'nota': 'delta %+.4f (5 restricoes)' % (_f12 - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T12 monotonicas: %.4f' % _f12, flush=True)


In [ ]:
# C3 — combo: target-mean suavizado (T9) + pairwise-15 (T10). Representacao x decisao.
from sklearn.utils import compute_sample_weight as _csw
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
_glob = y[itr].mean(); _m = 100.0
_tm = np.zeros((len(tr), 3))
for j_, c_ in enumerate(['MARSTALEGAL', 'MAINSTAT', 'HATLEVEL']):
    _g = pd.Series(y[itr]).groupby(t01[c_].iloc[itr].values)
    _tm[:, j_] = t01[c_].map(((_g.sum() + _m * _glob) / (_g.size() + _m))).fillna(_glob).to_numpy()
_drop = [c for c in COLS if any(c.startswith(p + '_') for p in ['MARSTALEGAL', 'MAINSTAT'])]
_keep = [c for c in COLS if c not in _drop]
Xc3 = np.hstack([tr[_keep].to_numpy(dtype=np.float64), _tm])
_mb = xgb_recipe(); _mb.fit(Xc3[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_pb = _mb.predict_proba(Xc3[ite]); _pa = _mb.predict(Xc3[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15)]:
    mtr = np.isin(y[itr], [a_, b_]); yb = (y[itr][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xc3[itr][mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xc3[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_f = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'C3 target-mean + pairwise15', 'f1': round(float(_f), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f (barra +0.015)' % (_n, _f - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('C3: %.4f' % _f, flush=True)


In [ ]:
# C4 — combo: base monotonica (T12) + pairwise-15 (T10). Restricao x decisao local.
from sklearn.utils import compute_sample_weight as _csw
_mc = {c: 0 for c in COLS}
for c in ['N_DOENCAS', 'N_DIFICULDADES', 'IDADE_MINIMA', 'FAIXA_MINIMA']:
    if c in _mc: _mc[c] = 1
if 'HATLEVEL' in _mc: _mc['HATLEVEL'] = -1
_mono = tuple(_mc[c] for c in COLS)
_m4 = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                    monotone_constraints=_mono, random_state=RANDOM_STATE,
                    n_jobs=-1, eval_metric='mlogloss')
_m4.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_pb = _m4.predict_proba(Xv[ite]); _pa = _m4.predict(Xv[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15)]:
    mtr = np.isin(y[itr], [a_, b_]); yb = (y[itr][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xv[itr][mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_f = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'C4 monotonica + pairwise15', 'f1': round(float(_f), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f (barra +0.015)' % (_n, _f - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('C4: %.4f' % _f, flush=True)


In [ ]:
# C5 — combo: quarentena honesta (T6) + pairwise-15 (T10). Distribuicao x decisao.
from sklearn.utils import compute_sample_weight as _csw
from sklearn.model_selection import StratifiedKFold as _SKF
_in = list(_SKF(4, shuffle=True, random_state=43).split(Xv[itr], y[itr]))
_conf = np.zeros(len(itr))
for _a, _b in _in:
    _mq = xgb_recipe()
    _mq.fit(Xv[itr][_a], y[itr][_a], sample_weight=_csw('balanced', y[itr][_a]))
    _conf[_b] = _mq.predict_proba(Xv[itr][_b]).max(1)
_qk = np.argsort(_conf)[int(len(itr) * 0.10):]
Xq, yq = Xv[itr][_qk], y[itr][_qk]
_mb = xgb_recipe(); _mb.fit(Xq, yq, sample_weight=_csw('balanced', yq))
_pb = _mb.predict_proba(Xv[ite]); _pa = _mb.predict(Xv[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15)]:
    mtr = np.isin(yq, [a_, b_]); yb = (yq[mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xq[mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_f = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'C5 quarentena + pairwise15', 'f1': round(float(_f), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f (barra +0.015)' % (_n, _f - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('C5: %.4f' % _f, flush=True)


In [ ]:
# T13 — especialistas de confusao: binarios treinados SO nas linhas que a base confunde
# (top-2 == par na OOF interna), nao em todas as linhas do par. Fronteira, nao distribuicao.
from sklearn.utils import compute_sample_weight as _csw
from sklearn.model_selection import StratifiedKFold as _SKF
_in = list(_SKF(2, shuffle=True, random_state=44).split(Xv[itr], y[itr]))
_oof_t2 = np.zeros((len(itr), 2), dtype=int)
for _a, _b in _in:
    _ma = xgb_recipe(); _ma.fit(Xv[itr][_a], y[itr][_a], sample_weight=_csw('balanced', y[itr][_a]))
    _oof_t2[_b] = np.sort(np.argsort(_ma.predict_proba(Xv[itr][_b]), axis=1)[:, -2:], axis=1)
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_pb = _mb.predict_proba(Xv[ite]); _pa = _mb.predict(Xv[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15)]:
    cmask = (_oof_t2[:, 0] == a_) & (_oof_t2[:, 1] == b_)
    if cmask.sum() < 50:
        continue
    yb = (y[itr][cmask] == b_).astype(int)
    if yb.mean() in (0.0, 1.0):
        continue
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xv[itr][cmask], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_f = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'T13 especialistas de confusao', 'f1': round(float(_f), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f (barra +0.010)' % (_n, _f - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T13 confusion specialists: %.4f (override %d)' % (_f, _n), flush=True)


In [ ]:
# T14 — pairwise-15 com gate de margem: so override se top-2 adjacente E pmax < 0.6
# (nao mexer nas confiantes; variante-B da E2 no harness atual).
from sklearn.utils import compute_sample_weight as _csw
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_pb = _mb.predict_proba(Xv[ite]); _pa = _mb.predict(Xv[ite])
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
_maxp = _pb.max(1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15)]:
    mtr = np.isin(y[itr], [a_, b_]); yb = (y[itr][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xv[itr][mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_) & (_maxp < 0.6)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_f = f1_score(y[ite], _pred, average='macro')
RESULTADOS.append({'variante': 'T14 pairwise + gate pmax<0.6', 'f1': round(float(_f), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f (barra +0.010)' % (_n, _f - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T14 margin-gated: %.4f (override %d)' % (_f, _n), flush=True)


In [ ]:
# S1 / S1b — shortlist #1: receita sub7 (3 familias ordinais + mediana@0.40 + correcao
# FAIXA_MINIMA) com e sem o override pairwise-15. T=0.40 pre-declarado (colega: escolhido
# em 4/5 partes), sem tuning no holdout. Pares adjacentes = lista completa, pre-declarada.
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils import compute_sample_weight as _csw

def _fh(criar, Xtr, ytr, Xpv):
    """Frank & Hall: 15 binarios 'faixa > k' (pesos balanced das 16 faixas) -> proba."""
    pesos = _csw('balanced', ytr)
    acima = np.zeros((len(Xpv), 15))
    for k in range(15):
        m = criar()
        m.fit(Xtr, (ytr > k).astype(int), sample_weight=pesos)
        acima[:, k] = m.predict_proba(Xpv)[:, 1]
    acima = np.minimum.accumulate(acima, axis=1)
    pr = np.clip(np.column_stack([1 - acima[:, 0], acima[:, :-1] - acima[:, 1:], acima[:, -1]]), 1e-6, None)
    return pr / pr.sum(axis=1, keepdims=True)

_XGB300 = lambda sd: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
    min_child_weight=3, subsample=0.8, colsample_bytree=0.5, tree_method='hist',
    eval_metric='logloss', n_jobs=-1, random_state=sd)
_LGB300 = lambda sd: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
    min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
    reg_lambda=1, verbose=-1, n_jobs=-1, random_state=sd)
_HGB300 = lambda sd: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
    max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1,
    early_stopping=False, random_state=sd)

_t0s = time.time()
_ord = np.mean([_fh(lambda s=s: _XGB300(s), Xv[itr], y[itr], Xv[ite]) for s in [42, 1, 2, 3, 4]], axis=0)
print('S1 XGB-ordinal 5 sementes: %.0fs' % (time.time() - _t0s), flush=True)
_lgb = np.mean([_fh(lambda s=s: _LGB300(s), Xv[itr], y[itr], Xv[ite]) for s in [42, 1, 2, 3]], axis=0)
print('S1 LGBM-ordinal 4 sementes: %.0fs' % (time.time() - _t0s), flush=True)
_hgb = np.mean([_fh(lambda s=s: _HGB300(s), Xv[itr], y[itr], Xv[ite]) for s in [42, 1]], axis=0)
_p3 = (_ord + _lgb + _hgb) / 3
print('S1 3 familias prontas: %.0fs' % (time.time() - _t0s), flush=True)

def _med(pr):
    return (np.cumsum(pr, axis=1) < 0.5).sum(axis=1)

fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
_pred = np.maximum(np.where(_p3.max(1) >= 0.40, _p3.argmax(1), _med(_p3)), fmin[ite])
_f1 = f1_score(y[ite], _pred, average='macro')

# S1b: override pairwise-15 por cima da receita (top-2 do proba medio da receita)
_top2 = np.sort(np.argsort(_p3, axis=1)[:, -2:], axis=1)
_pv = _pred.copy(); _n = 0
for a_, b_ in [(k, k + 1) for k in range(15)]:
    mtr = np.isin(y[itr], [a_, b_]); yb = (y[itr][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xv[itr][mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
    if sel.sum():
        _pv[sel] = np.where(m.predict_proba(Xv[ite][sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
_pv = np.maximum(_pv, fmin[ite])
_f2 = f1_score(y[ite], _pv, average='macro')

RESULTADOS.append({'variante': 'S1 receita sub7 (mediana@0.40)', 'f1': round(float(_f1), 4),
                   'tempo_s': round(time.time() - _t0s, 1), 'nota': 'delta %+.4f vs BASE' % (_f1 - BASE)})
RESULTADOS.append({'variante': 'S1b sub7 + pairwise15', 'f1': round(float(_f2), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f vs BASE, %+.4f vs S1' % (_n, _f2 - BASE, _f2 - _f1)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-2:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-2:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('S1 receita sub7: %.4f | S1b +pairwise15: %.4f (override %d) | %.0fs' % (_f1, _f2, _n, time.time() - _t0s), flush=True)

In [ ]:
# T15 — shortlist #3: bagging de sementes + linhas. 10 seeds (42..51), bootstrap 80%
# das linhas do treino por seed, media das probabilidades -> argmax. Sem tuning.
from sklearn.utils import compute_sample_weight as _csw
_t0 = time.time()
_bag = np.zeros((len(ite), K))
for _s in range(42, 52):
    _rng = np.random.RandomState(_s)
    _idx = _rng.choice(itr, size=int(len(itr) * 0.8), replace=True)
    _mb = xgb_recipe(sd=_s)
    _mb.fit(Xv[_idx], y[_idx], sample_weight=_csw('balanced', y[_idx]))
    _bag += _mb.predict_proba(Xv[ite]) / 10.0
_f15 = f1_score(y[ite], _bag.argmax(1), average='macro')
RESULTADOS.append({'variante': 'T15 bagging 10 seeds x 80% linhas', 'f1': round(float(_f15), 4),
                   'tempo_s': round(time.time() - _t0, 1), 'nota': 'delta %+.4f' % (_f15 - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T15 bagging: %.4f (%.0fs)' % (_f15, time.time() - _t0), flush=True)

In [ ]:
# T16 / T16b — shortlist #4: membro lambdarank (LGBMRanker, label_gain quadratico 0..15).
# v1 deu 0.1131 (predicao constante): escalas de score entre modelos interno e full nao
# sao comparaveis -> todos os scores do teste caem fora do intervalo de treino.
# v2: score -> percentil DENTRO da distribuicao de scores de treino do MESMO modelo;
# pareamento (percentil, y) continua OOF (2-fold interno) para nao otimizar no holdout.
# VEREDITO (10 nov): REJEITADO — v2 = 0.1005. Diagnostico: corr(pct_oof, y) = 0.66 mas
# o mapeio OOF so da F1 0.4133 no treino (membro fraco, nao esta mal ligado); e o blend
# w=0.5 com membro one-hot e degenerado por construcao (0.5 + 0.5p[c] >= 0.5p[d] sempre)
# -> T16b === T16. Sem retry: esperado do shortlist +0.000..0.004, medido << barra.
from lightgbm import LGBMRanker
from sklearn.model_selection import StratifiedKFold as _SKF
from sklearn.utils import compute_sample_weight as _csw
_t0 = time.time()
_gain = [float(i * i) for i in range(16)]
_mk = lambda: LGBMRanker(objective='lambdarank', label_gain=_gain, n_estimators=300,
                         learning_rate=0.05, num_leaves=15, min_child_samples=20,
                         reg_lambda=1, verbose=-1, n_jobs=-1, random_state=RANDOM_STATE)
def _pct(scores, ref):
    """percentil de cada score na distribuicao de ref (scores in-sample do MESMO modelo)."""
    r = np.sort(ref)
    return np.searchsorted(r, scores, side='right') / len(r)
_in = list(_SKF(2, shuffle=True, random_state=45).split(Xv[itr], y[itr]))
_s_oof = np.zeros(len(itr))
for _a, _b in _in:
    _mr = _mk(); _mr.fit(Xv[itr][_a], y[itr][_a], group=[len(_a)])
    _s_oof[_b] = _pct(_mr.predict(Xv[itr][_b]), _mr.predict(Xv[itr][_a]))
_mr = _mk(); _mr.fit(Xv[itr], y[itr], group=[len(itr)])
_s_te = _pct(_mr.predict(Xv[ite]), _mr.predict(Xv[itr]))
_ordi = np.argsort(_s_oof, kind='stable')
_y_sorted = y[itr][_ordi]; _s_sorted = _s_oof[_ordi]
_pred16 = _y_sorted[np.clip(np.searchsorted(_s_sorted, _s_te, side='left'), 0, len(_y_sorted) - 1)]
_f16 = f1_score(y[ite], _pred16, average='macro')
print('T16 diag: pred unicas %d | F1 %.4f | s_oof [%.3f,%.3f] s_te [%.3f,%.3f]' % (
    len(np.unique(_pred16)), _f16, _s_oof.min(), _s_oof.max(), _s_te.min(), _s_te.max()), flush=True)
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_p_x = _mb.predict_proba(Xv[ite])
_f16b = f1_score(y[ite], (0.5 * _p_x + 0.5 * np.eye(K)[_pred16]).argmax(1), average='macro')
RESULTADOS.append({'variante': 'T16 lambdarank mapeado', 'f1': round(float(_f16), 4),
                   'tempo_s': round(time.time() - _t0, 1), 'nota': 'delta %+.4f (pct normalizado, pareamento OOF)' % (_f16 - BASE)})
RESULTADOS.append({'variante': 'T16b lambdarank blend 0.5', 'f1': round(float(_f16b), 4),
                   'tempo_s': 0, 'nota': 'delta %+.4f vs BASE, %+.4f vs T16' % (_f16b - BASE, _f16b - _f16)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-2:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-2:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T16 lambdarank v2: %.4f | T16b blend: %.4f (%.0fs)' % (_f16, _f16b, time.time() - _t0), flush=True)

In [ ]:
# C6 — kitchen sink (tudo o que e positivo e compativel): tudo o que e positivo e compativel, junto uma vez.
# T6 quarentena + T9 target-mean suavizado + T12 monotonica + T14 gated pairwise
# + correcao piso. Pre-registo: barra +0.015; se perder para o melhor single
# (T10 +0.0162), combinacoes fecham em definitivo.
from sklearn.utils import compute_sample_weight as _csw
from sklearn.model_selection import StratifiedKFold as _SKF
_in = list(_SKF(4, shuffle=True, random_state=43).split(Xv[itr], y[itr]))
_conf = np.zeros(len(itr))
for _a, _b in _in:
    _mq = xgb_recipe()
    _mq.fit(Xv[itr][_a], y[itr][_a], sample_weight=_csw('balanced', y[itr][_a]))
    _conf[_b] = _mq.predict_proba(Xv[itr][_b]).max(1)
_qk = np.argsort(_conf)[int(len(itr) * 0.10):]
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
_glob = y[itr][_qk].mean(); _m = 100.0
_tm_tr = np.zeros((len(_qk), 3)); _tm_te = np.zeros((len(ite), 3))
for j_, c_ in enumerate(['MARSTALEGAL', 'MAINSTAT', 'HATLEVEL']):
    _g = pd.Series(y[itr][_qk]).groupby(t01[c_].iloc[itr].iloc[_qk].values)
    _sm = ((_g.sum() + _m * _glob) / (_g.size() + _m))
    _tm_tr[:, j_] = t01[c_].iloc[itr].iloc[_qk].map(_sm).fillna(_glob).to_numpy()
    _tm_te[:, j_] = t01[c_].iloc[ite].map(_sm).fillna(_glob).to_numpy()
_drop = [c for c in COLS if any(c.startswith(p + '_') for p in ['MARSTALEGAL', 'MAINSTAT'])]
_keep = [c for c in COLS if c not in _drop]
Xtr = np.hstack([tr[_keep].to_numpy(dtype=np.float64)[itr][_qk], _tm_tr])
Xte = np.hstack([tr[_keep].to_numpy(dtype=np.float64)[ite], _tm_te])
_mc = {c: 0 for c in list(_keep) + ['TM_MARST', 'TM_MAINST', 'TM_HAT']}
for c in ['N_DOENCAS', 'N_DIFICULDADES', 'IDADE_MINIMA', 'FAIXA_MINIMA']:
    if c in _mc: _mc[c] = 1
if 'HATLEVEL' in _mc: _mc['HATLEVEL'] = -1
_mono = tuple(_mc[c] for c in list(_keep) + ['TM_MARST', 'TM_MAINST', 'TM_HAT'])
_mq = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                    monotone_constraints=_mono, random_state=RANDOM_STATE,
                    n_jobs=-1, eval_metric='mlogloss')
_mq.fit(Xtr, y[itr][_qk], sample_weight=_csw('balanced', y[itr][_qk]))
_pb = _mq.predict_proba(Xte); _pa = _mq.predict(Xte)
_top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1); _maxp = _pb.max(1)
_pred = _pa.copy(); _n = 0
for a_, b_ in [(k, k + 1) for k in range(15)]:
    mtr = np.isin(y[itr][_qk], [a_, b_]); yb = (y[itr][_qk][mtr] == b_).astype(int)
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                      subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                      random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    m.fit(Xtr[mtr], yb, sample_weight=_csw('balanced', yb))
    sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_) & (_maxp < 0.6)
    if sel.sum():
        _pred[sel] = np.where(m.predict_proba(Xte[sel])[:, 1] >= 0.5, b_, a_)
        _n += int(sel.sum())
fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
_predc = np.maximum(_pred, fmin[ite])
_f15 = f1_score(y[ite], _predc, average='macro')
RESULTADOS.append({'variante': 'C6 kitchen sink', 'f1': round(float(_f15), 4),
                   'tempo_s': 0, 'nota': 'override %d | delta %+.4f (barra +0.015)' % (_n, _f15 - BASE)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('C6 kitchen sink: %.4f (override %d)' % (_f15, _n), flush=True)


In [ ]:
# T17 / T17b — shortlist #5 desbloqueado (licenca TabPFN aceite 12 nov): TabPFN-3.5 como
# familia extra, unico vies indutivo novo. CPU: n_estimators=2 fixo (knob de calculo, nao
# tuning — pre-declarado); barra +0.010. T17 sozinho (informativo) | T17b blend de PROBAS
# 0.5 XGB + 0.5 TabPFN (mistura valida; diferente do blend degenerado do T16).
# Nota custo: inferencia TabPFN em CPU e cara (smoke: 800 treino -> 961 s p/ 2193 test).
# Cap CPU de 5000 amostras: IGNORADO de proposito (ignore_pretraining_limits=True) — e um
# guard de performance, nao de correcao, e a regra do NB18 exige 100% das linhas para decisao.
from tabpfn import TabPFNClassifier
from sklearn.utils import compute_sample_weight as _csw
_t0 = time.time()
_tp = TabPFNClassifier(device='cpu', n_estimators=2, random_state=RANDOM_STATE,
                       show_progress_bar=True, ignore_pretraining_limits=True)
_tp.fit(Xv[itr], y[itr])
_ptp = _tp.predict_proba(Xv[ite])
_f17 = f1_score(y[ite], _ptp.argmax(1), average='macro')
print('T17 TabPFN fit+predict: %.0fs' % (time.time() - _t0), flush=True)
_mb = xgb_recipe(); _mb.fit(Xv[itr], y[itr], sample_weight=_csw('balanced', y[itr]))
_px = _mb.predict_proba(Xv[ite])
_f17b = f1_score(y[ite], (0.5 * _px + 0.5 * _ptp).argmax(1), average='macro')
# diagnostico de overlap de erros (porquê de um blend funcionar ou nao)
_eT = _ptp.argmax(1) != y[ite]
_eX = _px.argmax(1) != y[ite]
_both = float((_eT & _eX).mean()); _agree = float((_ptp.argmax(1) == _px.argmax(1)).mean())
print('overlap: ambos erram %.1f%% | concordam %.1f%%' % (100 * _both, 100 * _agree), flush=True)
RESULTADOS.append({'variante': 'T17 TabPFN-3.5 sozinho (n_est=2)', 'f1': round(float(_f17), 4),
                   'tempo_s': round(time.time() - _t0, 1),
                   'nota': 'delta %+.4f vs BASE | overlap ambos-erram %.1f%%' % (_f17 - BASE, 100 * _both)})
RESULTADOS.append({'variante': 'T17b blend 0.5 XGB+TabPFN', 'f1': round(float(_f17b), 4),
                   'tempo_s': 0,
                   'nota': 'delta %+.4f vs BASE, %+.4f vs T17 | concord %.1f%%' % (_f17b - BASE, _f17b - _f17, 100 * _agree)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-2:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-2:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('T17: %.4f | T17b blend: %.4f (%.0fs total)' % (_f17, _f17b, time.time() - _t0), flush=True)

In [ ]:
# S2 — receita sub7 + TabPFN (shortlist #5, ultimo passo do ecrã): media de PROBAS
# pre-declarada 0.5 (sem tuning de peso), receita igual à S1 (mediana@0.40 + correcao
# piso). Cache data/nb18_tabpfn_proba.npz: o fit caro do TabPFN reutiliza-se entre corridas.
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils import compute_sample_weight as _csw
from tabpfn import TabPFNClassifier
_t0 = time.time()

def _fh(criar, Xa, ya, Xb):
    """Frank & Hall: 15 binarios 'faixa > k' (pesos balanced das 16 faixas) -> proba."""
    pesos = _csw('balanced', ya)
    acima = np.zeros((len(Xb), 15))
    for k in range(15):
        m = criar()
        m.fit(Xa, (ya > k).astype(int), sample_weight=pesos)
        acima[:, k] = m.predict_proba(Xb)[:, 1]
    acima = np.minimum.accumulate(acima, axis=1)
    pr = np.clip(np.column_stack([1 - acima[:, 0], acima[:, :-1] - acima[:, 1:], acima[:, -1]]), 1e-6, None)
    return pr / pr.sum(axis=1, keepdims=True)

_XGB300 = lambda sd: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
    min_child_weight=3, subsample=0.8, colsample_bytree=0.5, tree_method='hist',
    eval_metric='logloss', n_jobs=-1, random_state=sd)
_LGB300 = lambda sd: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
    min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
    reg_lambda=1, verbose=-1, n_jobs=-1, random_state=sd)
_HGB300 = lambda sd: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
    max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1,
    early_stopping=False, random_state=sd)

_ord = np.mean([_fh(lambda s=s: _XGB300(s), Xv[itr], y[itr], Xv[ite]) for s in [42, 1, 2, 3, 4]], axis=0)
_lgb = np.mean([_fh(lambda s=s: _LGB300(s), Xv[itr], y[itr], Xv[ite]) for s in [42, 1, 2, 3]], axis=0)
_hgb = np.mean([_fh(lambda s=s: _HGB300(s), Xv[itr], y[itr], Xv[ite]) for s in [42, 1]], axis=0)
_p3 = (_ord + _lgb + _hgb) / 3
print('S2 receita 3 familias: %.0fs' % (time.time() - _t0), flush=True)

_cp = os.path.join(DATA_DIR, 'nb18_tabpfn_proba.npz')
if os.path.exists(_cp):
    _z = np.load(_cp)
    assert (_z['itr'] == itr).all() and (_z['ite'] == ite).all(), 'cache de outro split'
    _ptp = _z['ptp']
    print('S2 TabPFN proba lida do cache', flush=True)
else:
    _tp = TabPFNClassifier(device='cpu', n_estimators=2, random_state=RANDOM_STATE,
                           show_progress_bar=True, ignore_pretraining_limits=True)
    _tp.fit(Xv[itr], y[itr])
    _ptp = _tp.predict_proba(Xv[ite])
    np.savez(_cp, ptp=_ptp, itr=itr, ite=ite)
    print('S2 TabPFN fit+predict: %.0fs (cache gravado)' % (time.time() - _t0), flush=True)

def _med(pr):
    return (np.cumsum(pr, axis=1) < 0.5).sum(axis=1)
fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
def _regra(pr):
    return np.maximum(np.where(pr.max(1) >= 0.40, pr.argmax(1), _med(pr)), fmin[ite])

_f_s1 = f1_score(y[ite], _regra(_p3), average='macro')          # recomputo da S1 (referencia interna)
_f_s2 = f1_score(y[ite], _regra(0.5 * _p3 + 0.5 * _ptp), average='macro')
RESULTADOS.append({'variante': 'S2 receita sub7 + TabPFN (0.5)', 'f1': round(float(_f_s2), 4),
                   'tempo_s': round(time.time() - _t0, 1),
                   'nota': 'delta %+.4f vs BASE, %+.4f vs S1 recomputada %.4f (barra +0.010)' % (_f_s2 - BASE, _f_s2 - _f_s1, _f_s1)})
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)
print('S1 recomputada: %.4f | S2 receita+TabPFN: %.4f (%.0fs)' % (_f_s1, _f_s2, time.time() - _t0), flush=True)

## T18 — distâncias ao KMeans fold-safe (fecha E22)

Três features: `KM_DMIN` (distância ao centroide mais próximo), `KM_DMEAN` (distância média aos 16 centroides) e `KM_SIZE` (tamanho do cluster mais próximo). O scaler e o KMeans ajustam-se só a `itr`. Não entram os 16 one-hot dos clusters. A barra é +0,010.


In [ ]:
# T18 — KMeans fold-safe: distancias ao centroide mais proximo (fecha E22)
# KM_DMIN, KM_DMEAN e KM_SIZE. O scaler e o KMeans ajustam-se so a itr.
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

_sc = StandardScaler().fit(Xv[itr])
_km = KMeans(n_clusters=16, n_init=10, random_state=RANDOM_STATE).fit(_sc.transform(Xv[itr]))
_cent = _km.cluster_centers_
_size = np.bincount(_km.labels_, minlength=16)

def _dist_feats(Xa):
    d = cdist(_sc.transform(Xa), _cent)
    j = d.argmin(axis=1)
    return np.column_stack([d.min(axis=1), d.mean(axis=1), _size[j]])

X18 = np.column_stack([Xv, _dist_feats(Xv)])
avaliar('T18 KMeans dist fold-safe', X18[itr], X18[ite],
        nota='KM_DMIN+KM_DMEAN+KM_SIZE; barra +0.010')
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


### T18 — veredito

O T18 deu 0,4691 contra 0,4700 do T0, um delta de −0,0009. Fica abaixo da barra de +0,010. As distâncias ao KMeans não acrescentam sinal ao XGB. A E22 fecha sem adoção.


### T18b — proxy PCA do autoencoder

O erro de reconstrução do PCA, calculado com validação interna de 3 folds no treino e ajustado no treino para o holdout, entra como uma feature. É o proxy barato antes de treinar a rede. A barra é +0,010.


In [ ]:
# T18b — proxy PCA do autoencoder: erro de reconstrucao cross-fitted como feature
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold

_scp = StandardScaler().fit(Xv[itr])
_A = _scp.transform(Xv[itr])
_B = _scp.transform(Xv[ite])
_err = np.zeros(len(Xv))
for _ia, _ib in StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE).split(_A, y[itr]):
    _p = PCA(n_components=0.90, random_state=RANDOM_STATE).fit(_A[_ia])
    _err[itr[_ib]] = np.sqrt(((_A[_ib] - _p.inverse_transform(_p.transform(_A[_ib]))) ** 2).mean(1))
_p = PCA(n_components=0.90, random_state=RANDOM_STATE).fit(_A)
_err[ite] = np.sqrt(((_B - _p.inverse_transform(_p.transform(_B))) ** 2).mean(1))

X18b = np.column_stack([Xv, _err])
avaliar('T18b PCA erro de reconstrucao', X18b[itr], X18b[ite],
        nota='proxy do autoencoder; barra +0.010')
_ant = pd.read_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'))
_ant = _ant[~_ant['variante'].isin([r['variante'] for r in RESULTADOS[-1:]])]
pd.concat([_ant, pd.DataFrame(RESULTADOS[-1:])], ignore_index=True).sort_values('f1', ascending=False).to_csv(os.path.join(DATA_DIR, 'nb18_resultados.csv'), index=False)


### T18b — veredito

O T18b deu 0,4736 contra 0,4700, um delta de +0,0036. Fica abaixo da barra de +0,010. O erro de reconstrução do PCA não acrescenta sinal ao XGB; o autoencoder MLP não foi treinado.
